# NB5 ? Alpha-AS-1 Double DQN

This notebook adds a paper-style Alpha-AS-1 experiment inspired by Falces Mar?n et al. The key difference from B1/B2/B3 is structural: the DQN does **not** output bid/ask depths directly. It selects one of 20 `(gamma, skew)` actions every 5 seconds; the Avellaneda-Stoikov formula then computes the actual quotes.

Source reference: Falces Mar?n et al., *A reinforcement learning approach to improve the performance of the Avellaneda-Stoikov market-making algorithm*, PLOS ONE, 2022: https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0277042

Implementation limitation: this is adapted to the current DOGE snapshot simulator. The available data do not exactly reproduce the paper's full event/trade feed, so the notebook should be treated as a faithful architecture adaptation, not an exact replication.

In [1]:
import sys
import pathlib
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = next(
    (p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
     if (p / "procs").exists()),
    pathlib.Path.cwd(),
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from procs.gym.experiment_config import ReplayExperimentConfig
from procs.gym.data_loader import load_multi_day
from procs.gym.calibration import calibrate_from_arrays
from procs.gym.alpha_as import (
    AlphaASReplayEnv,
    DoubleDQNConfig,
    build_alpha_as_action_grid,
    train_double_dqn,
    evaluate_alpha_as_policy,
)

cfg = ReplayExperimentConfig()
cfg.ensure_artifact_dirs()
print(f"Repo root : {cfg.repo_root}")
print(f"Datasets  : {cfg.datasets_dir}")
print(f"Models    : {cfg.models_dir}")
print(f"Results   : {cfg.results_dir}")

Repo root : C:\Users\john-\Documents\Thesis_AI4T
Datasets  : C:\Users\john-\Documents\Thesis_AI4T\datasets
Models    : C:\Users\john-\Documents\Thesis_AI4T\models
Results   : C:\Users\john-\Documents\Thesis_AI4T\results


In [2]:
TRAIN_DAYS = 6
EXPECTED_TEST_DAYS = 23

daily_S, daily_dt, dates = load_multi_day(str(cfg.datasets_dir), pair=cfg.pair)
train_S, train_dt, train_dates = daily_S[:TRAIN_DAYS], daily_dt[:TRAIN_DAYS], dates[:TRAIN_DAYS]
test_S, test_dt, test_dates = daily_S[TRAIN_DAYS:], daily_dt[TRAIN_DAYS:], dates[TRAIN_DAYS:]

if len(train_S) != TRAIN_DAYS or len(test_S) != EXPECTED_TEST_DAYS:
    raise ValueError(f"Expected {TRAIN_DAYS} train and {EXPECTED_TEST_DAYS} test days, found {len(train_S)} and {len(test_S)}.")
if set(map(str, train_dates)) & set(map(str, test_dates)):
    raise ValueError("Train/test date overlap detected.")

print(f"Training days: {train_dates[0]} -> {train_dates[-1]}")
print(f"Test days    : {test_dates[0]} -> {test_dates[-1]}")

  2025-01-01:  713,815 snapshots, σ=0.000021
  2025-01-02:  766,464 snapshots, σ=0.000035
  2025-01-03:  776,383 snapshots, σ=0.000047
  2025-01-04:  778,293 snapshots, σ=0.000045
  2025-01-05:  723,494 snapshots, σ=0.000031
  2025-01-06:  766,311 snapshots, σ=0.000035
  2025-01-07:  787,093 snapshots, σ=0.000062
  2025-01-08:  821,589 snapshots, σ=0.000052
  2025-01-09:  809,421 snapshots, σ=0.000046
  2025-01-10:  789,320 snapshots, σ=0.000045
  2025-01-11:  724,826 snapshots, σ=0.000023
  2025-01-12:  719,550 snapshots, σ=0.000022
  2025-01-13:  819,981 snapshots, σ=0.000046
  2025-01-14:  775,454 snapshots, σ=0.000038
  2025-01-15:  782,299 snapshots, σ=0.000047
  2025-01-16:  788,946 snapshots, σ=0.000048
  2025-01-17:  801,259 snapshots, σ=0.000061
  2025-01-18:  829,125 snapshots, σ=0.000074
  2025-01-19:  842,892 snapshots, σ=0.000096
  2025-01-20:  857,508 snapshots, σ=0.000115
  2025-01-21:  848,107 snapshots, σ=0.000112
  2025-01-22:  812,180 snapshots, σ=0.000044
  2025-01-

In [3]:
train_params = []
for S, dt, date in zip(train_S, train_dt, train_dates):
    sigma_day, A_day, kappa_day = calibrate_from_arrays(S, dt, tick_size=cfg.tick_size)
    train_params.append({"date": date, "sigma": sigma_day, "A": A_day, "kappa": kappa_day})

param_frame = pd.DataFrame(train_params).set_index("date")
sigma_train = float(param_frame["sigma"].median())
A_train = float(param_frame["A"].median())
kappa_train = float(param_frame["kappa"].median())
print("Train-only Alpha-AS environment parameters:")
print(f"  sigma={sigma_train:.6f}, A={A_train:.4f}, kappa={kappa_train:.0f}")
display(param_frame)

Train-only Alpha-AS environment parameters:
  sigma=0.000028, A=0.2344, kappa=13968


,sigma,A,kappa
date,,,
2025-01-01,0.000020,0.256479,39821.831357
2025-01-02,0.000027,0.279438,21646.417707
2025-01-03,0.000036,0.248103,11016.985413
2025-01-04,0.000036,0.196818,6183.921908
2025-01-05,0.000027,0.173302,13891.853983
2025-01-06,0.000028,0.220755,14043.237468


In [4]:
grid = build_alpha_as_action_grid()
pd.DataFrame([action.__dict__ for action in grid])

,index,gamma,skew
0,0,0.01,-0.10
1,1,0.01,-0.05
2,2,0.01,0.00
3,3,0.01,0.05
4,4,0.01,0.10
5,5,0.10,-0.10
6,6,0.10,-0.05
7,7,0.10,0.00
8,8,0.10,0.05
9,9,0.10,0.10


In [5]:
train_env_counter = {"i": 0}


def make_train_env():
    # Cycle through the six training days whenever the DQN starts a new episode.
    i = train_env_counter["i"]
    train_env_counter["i"] += 1
    day_idx = i % len(train_S)
    seed = cfg.evaluation_seed + i
    return AlphaASReplayEnv(
        train_S[day_idx],
        train_dt[day_idx],
        sigma=sigma_train,
        A=A_train,
        kappa=kappa_train,
        terminal_time=float(train_dt[day_idx].sum()),
        tick_size=cfg.tick_size,
        q_max=cfg.q_max,
        decision_interval_sec=5.0,
        seed=seed,
    )

# Increase total_steps for the final run. The default here is intentionally modest
# so the notebook can be smoke-tested before a long Snellius run.
dqn_config = DoubleDQNConfig(
    total_steps=50_000,
    replay_buffer_size=10_000,
    batch_size=64,
    learning_rate=1e-3,
    discount=0.95,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay_steps=20_000,
    learning_starts=1_000,
    train_frequency=1,
    target_update_frequency=2,
    seed=cfg.evaluation_seed,
)

model_path = cfg.model_path("alpha_as1_dqn.pt")
policy, history = train_double_dqn(make_train_env, dqn_config, model_path=model_path)
print(f"Saved Alpha-AS-1 DQN -> {model_path}")
print(f"Training updates: {history['train_updates']}")

Saved Alpha-AS-1 DQN -> C:\Users\john-\Documents\Thesis_AI4T\models\alpha_as1_dqn.pt
Training updates: 49001


In [6]:
history_path = cfg.result_path("alpha_as1_dqn_history.json")
with open(history_path, "w") as f:
    json.dump(history, f)
print(f"Saved history -> {history_path}")

Saved history -> C:\Users\john-\Documents\Thesis_AI4T\results\alpha_as1_dqn_history.json


In [7]:
rows = []
for S, dt, date in zip(test_S, test_dt, test_dates):
    def env_factory(seed, S=S, dt=dt):
        return AlphaASReplayEnv(
            S,
            dt,
            sigma=sigma_train,
            A=A_train,
            kappa=kappa_train,
            terminal_time=float(dt.sum()),
            tick_size=cfg.tick_size,
            q_max=cfg.q_max,
            decision_interval_sec=5.0,
            seed=seed,
        )

    metrics = evaluate_alpha_as_policy(
        policy,
        env_factory,
        rollouts=cfg.evaluation_rollouts,
        seed=cfg.evaluation_seed,
    )
    metrics["Day"] = str(date)
    rows.append(metrics)
    print(f"{date}: Sharpe={metrics['Sharpe']:.4f}, MaxDD={metrics['Max DD']:.6f}, PnL={metrics['Final PnL']:.6f}")

df_alpha_as1 = pd.DataFrame(rows).set_index("Day")
out_path = cfg.result_path("alpha_as1_test_results.csv")
df_alpha_as1.to_csv(out_path)
print(f"Saved Alpha-AS-1 test results -> {out_path}")
df_alpha_as1.head()

2025-01-07: Sharpe=-0.0108, MaxDD=0.487347, PnL=-0.309076
2025-01-08: Sharpe=-0.0067, MaxDD=0.303665, PnL=-0.178939
2025-01-09: Sharpe=-0.0099, MaxDD=0.312640, PnL=-0.233395
2025-01-10: Sharpe=0.0069, MaxDD=0.171494, PnL=0.154643
2025-01-11: Sharpe=0.0044, MaxDD=0.067124, PnL=0.050467
2025-01-12: Sharpe=0.0022, MaxDD=0.093300, PnL=0.026404
2025-01-13: Sharpe=-0.0052, MaxDD=0.359019, PnL=-0.124853
2025-01-14: Sharpe=0.0111, MaxDD=0.105569, PnL=0.213344


KeyboardInterrupt: 

## Interpretation Note

Use this model as an additional experiment, not as a replacement for B1-B3. If Alpha-AS-1 outperforms the direct PPO agents, that supports the methodological argument that RL benefits from the A-S inductive bias: the network learns parameter corrections rather than learning quotes from scratch.